In [48]:
import os
import re
import pandas as pd
import time
import datetime
from csv import reader
from dateutil import parser
import glob
from pathlib import Path
import duckdb
from dateutil.parser import parse
import warnings
import pandas_dedupe
import csv
import sys 
from datetime import datetime
import json
warnings.filterwarnings("ignore")

class Extraction:

    def __init__(self, folder_path , failed_folder_path , duplicateId_folder_path) -> None:
        self.folder_path = folder_path
        self.failed_folder_path = failed_folder_path
        self.duplicateId_folder_path = duplicateId_folder_path


    def saveJson(self,dictionary,filename) -> None:
        with open(filename + '.json', 'w') as fp:
            json.dump(dictionary,fp)


    def readJson(self):
        with open(latest_site_id + '.json', 'r') as fp:
            data = json.load(fp)
        latest_site_id = list(data.keys())[0]
        return data , latest_site_id


    def getFailityDbList(self):
        facility_db_list = [f for f in os.listdir(self.folder_path) if f.endswith('.sql')]
        return facility_db_list
    

    def dbTableList(self):
        table_list = ["art_ipt","art","person","weight","patient","art_current_status","art_who_stage","tb_treatment_current_status",
                      "patient_tpt_screening","patient_client_profile","tb","art_visit","patient_tb_screening","patient_tpt_screening",
                      "person_investigation","person_diagnosis"
                    ]
        return table_list


    def split_string(self,string):
        # Match all commas outside of quotes
        pattern = r",(?=(?:[^']*'[^']*')*[^']*$)"
        return re.split(pattern, string)
    

    def readTables(self,folder_path,filename, table_list):
        with open(os.path.join(folder_path, filename), "r", encoding='ISO-8859-1') as f:
            content = f.read()
        content = content.replace("Unprotected sex (sex without a condom)", "Unprotected sex without a condom")
        latest_site_id = []
        latest_timestamp = []
        first_timestamp = []
        version = []
        domain_event = content.split("/*!40000 ALTER TABLE `domain_event_entry` DISABLE KEYS */;")
        if "/*!40000 ALTER TABLE `domain_event_entry` ENABLE KEYS */" in domain_event[1]:
            domain_event = domain_event[1].split("/*!40000 ALTER TABLE `domain_event_entry` ENABLE KEYS */")
            facility_search = re.findall('ZW[a-zA-Z0-9]{6}',domain_event[0])
            time_serach = re.findall('[0-9]{4}-[0-9]{2}-[0-9]{2}T[0-9]{2}:[0-9]{2}', domain_event[0])
            version = re.findall('[0-9]{1}\\.[0-9]{1}\\.[0-9]{2}-Q{1}[0-9]{1}',domain_event[0])
        else:
            facility_search = re.findall('ZW[a-zA-Z0-9]{6}',domain_event[1])
            time_serach = re.findall('[0-9]{4}-[0-9]{2}-[0-9]{2}T[0-9]{2}:[0-9]{2}', domain_event[1])
            version = re.findall('[0-9]{1}\\.[0-9]{1}\\.[0-9]{2}-Q{1}[0-9]{1}',domain_event[1])
        if facility_search:
            latest_site_id = facility_search[-1]
            latest_timestamp = time_serach[-1]
            first_timestamp = time_serach[0]
            if len(version)==0:
                version = ""
            else:
                version = version[-1]
        db_tables = {}
        fact_tables = {}
        database_statements = re.split(r'(?<=\n)CREATE DATABASE', content)
        for statement in database_statements[1:]:
            table_names = []
            database_name = re.findall(r'`(\w+)`', statement)[0]
            # Use regular expressions to extract the table names
            pattern = re.compile(r"CREATE TABLE `(.+?)`", re.DOTALL)
            matches = pattern.findall(statement)
            # Loop through the matches and append the table names to the list
            for match in matches:
                table_names.append(match)
            # Create a dictionary of dataframes for the tables
            tables_dict = {}
            for table_name in table_names:
                if table_name not in table_list:
                    continue
                table_str = re.search(rf'CREATE TABLE `{table_name}`.+?ENGINE=InnoDB DEFAULT CHARSET=utf8;', statement, re.DOTALL)
                if table_str is None:
                    continue
                table_str  = table_str.group(0).strip()
                columns = re.findall(r'`(\w+)` (\w+)', table_str)
                column_names = [column[0] for column in columns if not column[0].startswith("fk")]
                data = re.findall(rf'INSERT INTO `{table_name}` VALUES \(.+?\);', statement, re.DOTALL)
                rows = {}
                if data:
                    values_list = []
                    for row in data:            
                        row = re.findall(r'\((.*?)\)', row)
                        values2 = [ re.split(r',(?=")', s) for s in row]
                        for item in values2:
                            item = item[0].replace('NULL', "'NULL'")
                            rec = [i.replace("'","").lstrip() for i in item.split("',")]
                            values_list.append(rec)
                     
                    for column in column_names:
                        coulum_data = []
                        column_index = column_names.index(column)
                        for record in values_list:
                            if column_index < len(record):
                                coulum_data.append(record[column_index])
                            else:
                                coulum_data.append("")  
                        rows[column] = coulum_data
                    table_df = pd.DataFrame(rows)
                    table_df = table_df.to_dict(orient="records")
                    tables_dict[table_name] = table_df
                else:
                    table_df = pd.DataFrame(columns=column_names)
                    tables_dict[table_name] = table_df
            db_tables[database_name] = tables_dict
        fact_tables[latest_site_id] = db_tables
        return latest_site_id, latest_timestamp ,first_timestamp, version,fact_tables, row


    def getFacilityDetails(self,mapping_file,latest_site_id):
        facility_name = mapping_file.loc[mapping_file['Facility ID'] == latest_site_id] ["Facility"].values
        if facility_name.size > 0:
            facility_name = facility_name[0]
            district_name = mapping_file.loc[mapping_file['Facility ID'] == latest_site_id] ["District"].values
            if district_name.size > 0:
                district_name = district_name[0]
            else:
                district_name = ""
            province_name = mapping_file.loc[mapping_file['Facility ID'] == latest_site_id] ["Province"].values
            if province_name.size > 0:
                province_name = province_name[0]
            else:
                province_name = ""
        else:
            facility_name = ""
            district_name = ""
            province_name = ""
        return facility_name,district_name,province_name
    

    def getReasonforNotEiligible1(self,row):
            if row["Eligible for TPT at enrolment"] == "No":
                if row["sign_and_symptoms_of_active_tb"] == "Yes":
                        return "Sign and symptoms of active tb"
                elif row["patient_currently_on_tb_treatment"] =="Yes":
                        return "Patient currently on tb treatment"
                elif row["completed_ipt_in_the_last_three_years"] == "Yes":
                        return "Completed ipt in the last three years"
                elif row["signs_of_active_liver_disease"] == "Yes":
                        return "Signs of active liver disease"
                elif row["heavy_alcohol_use"] == "Yes":
                        return "Heavy alcohol use"
                elif row["severe_peripheral_neuropathy"] == "Yes":
                        return "Severe peripheral neuropathy"
            else:
                return ""

    def getReasonforNotEiligible(self,row):
            if row["Eligible for TPT"] == "No":
                if row["sign_and_symptoms_of_active_tb"] == "Yes":
                        return "Sign and symptoms of active tb"
                elif row["patient_currently_on_tb_treatment"] =="Yes":
                        return "Patient currently on tb treatment"
                elif row["completed_ipt_in_the_last_three_years"] == "Yes":
                        return "Completed ipt in the last three years"
                elif row["signs_of_active_liver_disease"] == "Yes":
                        return "Signs of active liver disease"
                elif row["heavy_alcohol_use"] == "Yes":
                        return "Heavy alcohol use"
                elif row["severe_peripheral_neuropathy"] == "Yes":
                        return "Severe peripheral neuropathy"
            else:
                return ""
   

    def getMappingFile(self):
        mapping_file = pd.read_csv("mapping_file.csv")
        mapping_file['Facility ID'] = mapping_file['Facility ID'].str.strip()
        return mapping_file

    
if __name__ == '__main__':
    processing_time = datetime.now().strftime("%d/%m/%Y %H:%M:%S")
    extraction  = Extraction("./", "./Failed/","./DuplicateId/")
    mapping_file = extraction.getMappingFile()
    table_list = extraction.dbTableList()
    facilities = extraction.getFailityDbList()
    folder_path = extraction.folder_path
  
    for filename in facilities:
        print()
        print("........................................................................")
        print(">>> working on ", filename)
        latest_site_id, latest_timestamp ,first_timestamp, version,fact_tables, row = extraction.readTables(folder_path,filename,table_list)

        facility_name , district_name,province_name =  extraction.getFacilityDetails(mapping_file,latest_site_id)
        print(">>> ",facility_name)

        art = pd.DataFrame(fact_tables[latest_site_id]["consultation"]["art"])
        person_investigation = pd.DataFrame(fact_tables[latest_site_id]["consultation"]["person_investigation"])
        art_ipt = pd.DataFrame(fact_tables[latest_site_id]["consultation"]["art_ipt"])
        weight = pd.DataFrame(fact_tables[latest_site_id]["consultation"]["weight"])
        patient = pd.DataFrame(fact_tables[latest_site_id]["consultation"]["patient"])
        art_who_stage = pd.DataFrame(fact_tables[latest_site_id]["consultation"]["art_who_stage"])
        art_current_status = pd.DataFrame(fact_tables[latest_site_id]["consultation"]["art_current_status"])
        tb = pd.DataFrame(fact_tables[latest_site_id]["consultation"]["tb"])
        tb_treatment_current_status = pd.DataFrame(fact_tables[latest_site_id]["consultation"]["tb_treatment_current_status"])
        demographics = pd.DataFrame(fact_tables[latest_site_id]["client"]["person"])
        art_visit = pd.DataFrame(fact_tables[latest_site_id]["consultation"]["art_visit"])
        tb_screening = pd.DataFrame(fact_tables[latest_site_id]["consultation"]["patient_tb_screening"])
        tpt_screening = pd.DataFrame(fact_tables[latest_site_id]["consultation"]["patient_tpt_screening"])
        person_diagnosis = pd.DataFrame(fact_tables[latest_site_id]["consultation"]["person_diagnosis"])
        if "patient_client_profile" in fact_tables[latest_site_id]["consultation"]:
            patient_client_profile = pd.DataFrame(fact_tables[latest_site_id]["consultation"]["patient_client_profile"]) 
        else:
            patient_client_profile = pd.DataFrame({"client_profile":[],"person_id":[]})
        

        print(">>> Facility id ",latest_site_id)


        # Correct wrongly formatted weight data
        weight["value2"] = weight["value"]
        weight["value"] = weight["value"].map(lambda x: x.split(",")[0])
        weight["patient_id"] = weight["value2"].map(lambda x: x.split(",")[1])
        weight.drop("value2",inplace= True, axis=1)


        df_art_start = duckdb.query(
            """
                select demographics.person_id,
                demographics.firstname,
                demographics.lastname,
                demographics.sex,
                demographics.birthdate,
                art.art_number,
                art.date_of_hiv_test as 'Date of HIV diagnosis',
                art.date_enrolled as 'Date of enrolmment in care'
                from art
                left join demographics
                on art.person_id = demographics.person_id
            """
        ).df()



        df_art_current_start = duckdb.query(
            """
                with ranked_messages as (
                    select 
                    art.person_id,
                    art_current_status.regimen as 'ARV Regimen on enrolemnt',
                    art_current_status.art_initiation_category as 'ART initiation category on enrolemnt',
                    art_current_status.date as 'ART start date',
                    ROW_NUMBER() OVER (PARTITION BY art.person_id ORDER BY art_current_status.date ASC) as rtt
                    from art_current_status
                    left join art 
                    on art_current_status.art_id = art.art_id
                )
                select * from ranked_messages where rtt = 1
            """
        ).df() 
        df_art_current_start.drop(["rtt"],inplace= True, axis=1)


        df_tpt_eligibility = duckdb.query(
            """
                WITH ranked_messages AS(
                    select 
                    patient.person_id,
                    patient.time as 'Event_date',
                    tpt_screening.eligible as 'Eligible for TPT at enrolment',
                    sign_and_symptoms_of_active_tb,patient_currently_on_tb_treatment,
                    completed_ipt_in_the_last_three_years,signs_of_active_liver_disease,heavy_alcohol_use,severe_peripheral_neuropathy,
                    ROW_NUMBER() OVER (PARTITION BY patient.person_id ORDER BY patient.time ASC) as rtt
                    from tpt_screening 
                    left join patient
                    on tpt_screening.patient_id = patient.patient_id
                    where patient.person_id in (select person_id from df_art_start)
                    )
                    select * from ranked_messages where rtt = 1
            """
        ).df()
        df_tpt_eligibility.drop(["rtt","Event_date"],inplace= True, axis=1)
        df_tpt_eligibility = df_tpt_eligibility.replace('\x01', 'Yes')
        df_tpt_eligibility = df_tpt_eligibility.replace('\\0', 'No')


        df_tpt_eligibility["Reason for not being eligible for TPT"] = df_tpt_eligibility.apply(extraction.getReasonforNotEiligible1, axis =1)

        df_tpt_eligibility = df_tpt_eligibility[[
            'person_id', 'Eligible for TPT at enrolment', 'Reason for not being eligible for TPT'
        ]]


        df_tpt_not_initiate = duckdb.query(
            """
                WITH ranked_messages AS(
                    select 
                    art.person_id,
                    art_ipt.date as 'Date reason given',
                    art_ipt.reason as 'Reason for not initiating TPT',
                    ROW_NUMBER() OVER (PARTITION BY art.person_id ORDER BY art_ipt.date DESC) as rtt
                    from art_ipt 
                    left join art
                    on art.art_id = art_ipt.art_id
                    where art_ipt.status = 'Not Eligible for IPT' and 
                    art.person_id in (select person_id from df_art_start)
                    )
                    select * from ranked_messages where rtt = 1
            """
        ).df()
        df_tpt_not_initiate.drop(["rtt"],inplace= True, axis=1)


        df_tpt_start = duckdb.query(
            """
                WITH ranked_messages AS(
                    select 
                    art.person_id,
                    art_ipt.date as 'Event_date',
                    art_ipt.date as "TPT start date", 
                    art_ipt.status as 'TPT Start Regimen',
                    ROW_NUMBER() OVER (PARTITION BY art.person_id ORDER BY art_ipt.date DESC) as rtt
                    from art_ipt 
                    left join art
                    on art.art_id = art_ipt.art_id
                    left join demographics on
                    art.person_id = demographics.person_id
                    where art_ipt.status in ('3HP Initiated', '3RH Initiated','6H Initiated') and 
                    art.person_id in (select person_id from df_art_start)
                    )
                    select * from ranked_messages where rtt = 1
            """
        ).df()
        df_tpt_start['TPT Start Regimen'] = df_tpt_start['TPT Start Regimen'].str.replace('Initiated','')
        df_tpt_start.drop(["rtt","Event_date"],inplace= True, axis=1)


        df_tpt_stop = duckdb.query(
            """
                WITH ranked_messages AS(
                    select 
                    art.person_id,
                    art.art_id,
                    art_ipt.date as 'Date of stopping TPT',
                    art_ipt.reason as 'Reason for stopping TPT',
                    ROW_NUMBER() OVER (PARTITION BY art.person_id ORDER BY art_ipt.date DESC) as rtt
                    from art_ipt 
                    left join art
                    on art.art_id = art_ipt.art_id
                    left join demographics on
                    art.person_id = demographics.person_id
                    where art_ipt.status in ('3HP Stopped', 'IPT Stopped','6H Stopped') and 
                    art.person_id in (select person_id from df_art_start)
                    )
                    select * from ranked_messages where rtt = 1
            """
        ).df()
        df_tpt_stop.drop(["rtt"],inplace= True, axis=1)


        df_tpt_complete = duckdb.query(
            """
                WITH ranked_messages AS(
                    select 
                    art.person_id,
                    art.art_id,
                    art_ipt.date as 'Event_date',
                    art_ipt.date as "TPT completion date", 
                    art_ipt.status as 'TPT Complete Regimen',
                    ROW_NUMBER() OVER (PARTITION BY art.person_id ORDER BY art_ipt.date DESC) as rtt
                    from art_ipt 
                    left join art
                    on art.art_id = art_ipt.art_id
                    left join demographics on
                    art.person_id = demographics.person_id
                    where art_ipt.status in ('IPT Completed','3HP Completed', '6H Completed') and 
                    art.person_id in (select person_id from df_art_start)
                    )
                    select * from ranked_messages where rtt = 1
            """
        ).df()
        df_tpt_complete['TPT Complete Regimen'] = df_tpt_complete['TPT Complete Regimen'].str.replace('Completed','')
        df_tpt_complete.drop(["rtt","Event_date"],inplace= True, axis=1)


        df_weight = duckdb.query(
            """
            with ranked_messages as (
                select  weight.time as 'Event_date',
                patient.person_id,
                weight.value as 'Weight at enrolment',
                ROW_NUMBER() OVER (PARTITION BY patient.person_id ORDER BY weight.time DESC) as rtt
                from weight
                left join patient
                on weight.patient_id = patient.patient_id
                where patient.person_id in (select person_id from df_art_start)
                )
                select * from ranked_messages where rtt = 1
            """
        ).df()
        df_weight.drop(["rtt","Event_date"],inplace= True, axis=1)


        df_art_who_stage = duckdb.query(
            """
            with ranked_messages as (
            select  art_who_stage.date as 'Event_date',
            art.person_id , art_who_stage.stage as 'WHO clinical stage at enrolment',
            art_who_stage.follow_up_status as 'Follow Up Outcome',
            ROW_NUMBER() OVER (PARTITION BY art.person_id ORDER BY art_who_stage.date ASC) as rtt
            from art_who_stage 
            left join art 
            on art_who_stage.art_id  = art.art_id
            where art.person_id in (select person_id from df_art_start)
            )
            select * from ranked_messages where rtt = 1
            """
        ).df()
        df_art_who_stage.drop(["rtt","Event_date"],inplace= True, axis=1)


        #  Get tb treatment outcome and outcome date
        df_TB = duckdb.query(
            """
                with ranked_messages as (
                select t.treatment_start_date as 'TB Treatment Start Date',
                pd.date as 'date of TB diagnosis', pd.diagnosis as 'TB diagnosis',
                tb.person_id ,tb.type_of_tb as 'Type Of TB',
                tb.number as 'TB number',tb.outcome as 'TB treatment outcome',
                tb.out_come_date as 'Date of TB outcome',
                ROW_NUMBER() OVER (PARTITION BY tb.person_id ORDER BY pd.date DESC) as rtt
                from tb
                left join tb_treatment_current_status t
                on tb.tb_id = t.tb_id
                left join person_diagnosis pd
                on tb.person_diagnosis_id = pd.person_diagnosis_id
                where  tb.person_id in (select tb.person_id from df_art_start)
                )
                select * from ranked_messages where rtt = 1
            """
        ).df()
        df_TB.drop(["rtt"],inplace= True, axis=1)


        df_art_visit = duckdb.query(
            """
            with ranked_messages as (
                 select art_visit.lactating_status as 'Pregnancy/lactation status at enrolment',
                 patient.person_id, patient.time,
                 ROW_NUMBER() OVER (PARTITION BY patient.person_id ORDER BY patient.time ASC) as rtt
                 from art_visit
                 left join patient
                 on art_visit.patient_id = patient.patient_id 
                )
                select * from ranked_messages where rtt = 1
            """
        ).df()
        df_art_visit.drop(["rtt"],inplace= True, axis=1)


        df_tb_screening = duckdb.query(
            """
            with ranked_messages as (
                 select patient.person_id,
                 tb_screening.presumptive as 'TB status at enrolment',
                 patient.time,
                 ROW_NUMBER() OVER (PARTITION BY patient.person_id ORDER BY patient.time DESC) as rtt
                 from tb_screening
                 left join patient
                 on tb_screening.patient_id = patient.patient_id
                )
                select * from ranked_messages where rtt = 1
            """
        ).df()
        df_tb_screening.drop(["rtt","time"],inplace= True, axis=1)
        df_tb_screening = df_tb_screening.replace('\x01', 'Suspect or Presumptive')
        df_tb_screening = df_tb_screening.replace('\\0', 'Screened with no signs')


        df_client_profile = duckdb.query(
            """
            with ranked_messages as (
                 select person_id,
                 client_profile as 'Client Profile',
                 date ,
                 ROW_NUMBER() OVER (PARTITION BY person_id ORDER BY date DESC) as rtt
                 from patient_client_profile
                )
                select * from ranked_messages where rtt = 1
            """
        ).df()
        df_client_profile.drop(["rtt","date"],inplace= True, axis=1)


        # Merging files
        merge1 = pd.merge(df_art_start , df_art_current_start, on = ["person_id"], how ="left")
        merge2 = pd.merge(merge1, df_weight, on = ["person_id"], how = "left")
        merge3 = pd.merge(merge2, df_tpt_start, on = ["person_id"], how = "left")
        merge4 = pd.merge(merge3, df_tpt_complete, on = ["person_id"], how = "left")
        merge5 = pd.merge(merge4, df_tpt_stop, on = ["person_id"], how = "left")
        merge6 = pd.merge(merge5, df_art_who_stage, on = ["person_id"] , how ="left" )
        merge7 = pd.merge(merge6, df_art_visit, on = ["person_id"], how = "left")
        merge8 = pd.merge(merge7, df_TB, on = ["person_id"] , how = "left")
        merge9 = pd.merge(merge8, df_tb_screening, on = ["person_id"], how = "left")
        merge10 = pd.merge(merge9, df_tpt_eligibility , on = ["person_id"] , how ="left")
        merge11 = pd.merge(merge10, df_tpt_not_initiate, on = ["person_id"] , how = "left")
        merge12 = pd.merge(merge11, df_client_profile, on =["person_id"], how ="left")
        merge12["Province"] = province_name
        merge12["District"] = district_name
        merge12["Facility"] = facility_name
        merge12["Facility Id"] = latest_site_id


        merge12 = merge12 [[
            'Province','District','Facility','Facility Id',
            'person_id','firstname','lastname','birthdate','sex', 'Client Profile','art_number','Date of HIV diagnosis',
            'Date of enrolmment in care','ART start date','ART initiation category on enrolemnt','ARV Regimen on enrolemnt',
            'Weight at enrolment', 'TB status at enrolment','WHO clinical stage at enrolment',
            'Pregnancy/lactation status at enrolment', 'Eligible for TPT at enrolment','Reason for not being eligible for TPT',
            'TPT start date','TPT Start Regimen','TPT completion date','TPT Complete Regimen',
            'Reason for stopping TPT', 'Date of stopping TPT','date of TB diagnosis','TB diagnosis',
            'TB Treatment Start Date','Type Of TB','TB treatment outcome','Date of TB outcome',
             'Follow Up Outcome'
        ]]
        static_file = merge12
        

        print(">>> working on sentinel file")

        ### Sentinel Events
        df_art_visit_sentinel = duckdb.query(
            """
                 select art_visit.lactating_status as 'Pregnancy/lactation status',
                 art_visit.tb_status as 'TB Status',
                 tb_screening.presumptive,
                 weight.value as 'Weight at ART Visit',
                 patient.person_id, patient.time as 'Event_date',
                 from art_visit
                 left join patient
                 on art_visit.patient_id = patient.patient_id 
                 left join weight
                 on patient.patient_id = weight.patient_id
                 left join tb_screening
                 on patient.patient_id = tb_screening.patient_id
                 left join tpt_screening
                 on patient.patient_id = tpt_screening.patient_id
                 where person_id in (select person_id from merge12)

            """
        ).df()
        df_art_visit_sentinel["Event_date"] = pd.to_datetime(df_art_visit_sentinel["Event_date"],errors='coerce').dt.date
        df_art_visit_sentinel.drop_duplicates( keep='last',inplace = True)

        df_art_visit_sentinel = df_art_visit_sentinel.replace('\x01', 'YES')
        df_art_visit_sentinel = df_art_visit_sentinel.replace('\\0', 'NO')


        df_art_current_status_sentinel = duckdb.query(
            """
                select a.date as 'Event_date', 
                a.date as 'Art Current Status Date',a.art_id , a.state as 'ARV Status', a.art_initiation_category as 'Art Initiation Category',
                art.person_id,a.regimen as 'ART Regimen'
                from art_current_status a 
                left join art 
                on a.art_id = art.art_id
                where art.person_id in (select person_id from merge12)
            """
        ).df()
        df_art_current_status_sentinel["Art Current Status Date"] = pd.to_datetime(df_art_current_status_sentinel["Art Current Status Date"],errors='coerce').dt.date
        df_art_current_status_sentinel["Event_date"] = pd.to_datetime(df_art_current_status_sentinel["Event_date"],errors='coerce').dt.date
        df_art_current_status_sentinel = df_art_current_status_sentinel[[
        'Event_date','person_id', 'ARV Status','ART Regimen','Art Initiation Category'
        ]]
        df_art_current_status_sentinel["Event_date"] = pd.to_datetime(df_art_current_status_sentinel["Event_date"],errors='coerce').dt.date
        df_art_current_status_sentinel.drop_duplicates(subset=['Event_date','person_id'], keep='last',inplace = True)


        #Art who stage..............................................................................................................
        df_art_who_stage_sentinel = duckdb.query(
            """
            select  a.date as 'Event_date',
            p.person_id , a.art_id, a.date as 'Who Stage Date',
            a.stage as 'Who Stage', a.follow_up_status as 'Art Follow Up Status'
            from art_who_stage a left join art p
            on a.art_id  = p.art_id
            where p.person_id in (select person_id from merge12)

            """
        ).df()

        df_art_who_stage_sentinel["Who Stage Date"] = pd.to_datetime(df_art_who_stage_sentinel["Who Stage Date"],errors='coerce').dt.date
        df_art_who_stage_sentinel["Event_date"] = pd.to_datetime(df_art_who_stage_sentinel["Event_date"],errors='coerce').dt.date
        df_art_who_stage_sentinel = df_art_who_stage_sentinel [[
            'Event_date',
            'person_id','Who Stage',
            'Art Follow Up Status'
        ]]

        df_art_who_stage_sentinel = df_art_who_stage_sentinel.sort_values(by=['Art Follow Up Status'])
        df_art_who_stage_sentinel.drop_duplicates(subset=['Event_date','person_id'], keep='last',inplace = True)


        df_viral_load_sentinel = duckdb.query(
            """
                select date as 'Event_date',
                person_id,date as "Date at which Viral Load result was issued",
                date as "Date for which Viral Load was taken",
                'TRUE' as "Viral Load Sample submitted to lab",
                'TRUE' as "Was Viral Load result issued",
                result as "Viral Load result",
                from person_investigation
                where test = 'Viral Load' and person_id in (select person_id from merge12)
            """
        ).df()
 
        df_viral_load_sentinel["Date at which Viral Load result was issued"] = pd.to_datetime(df_viral_load_sentinel["Date at which Viral Load result was issued"],errors='coerce').dt.date
        df_viral_load_sentinel["Event_date"] = pd.to_datetime(df_viral_load_sentinel["Event_date"],errors='coerce').dt.date
        df_viral_load_sentinel = df_viral_load_sentinel[[
            'Event_date',
            'person_id',
            "Date at which Viral Load result was issued",
            "Date for which Viral Load was taken",
            "Viral Load Sample submitted to lab",
            "Was Viral Load result issued",
            "Viral Load result"
        ]]

        df_viral_load_sentinel = df_viral_load_sentinel.sort_values(by=['Viral Load result'])
        df_viral_load_sentinel.drop_duplicates(subset=['Event_date','person_id'], keep='last', inplace = True)


        # cd4..............................................................................................................................
        df_cd4_sentinel = duckdb.query(
            """
                select date as 'Event_date',
                person_id, date as 'Date at which cd4 sample was taken',
                date as 'Date at which cd4 result was issued',
                'TRUE' as "CD4 Sample submitted to lab",
                'TRUE' as "Was cd4 result issued",
                result as 'CD4 Count'
                from person_investigation
                where test = 'CD4 Count' and person_id in (select person_id from merge12)
            """
        ).df()
   
        df_cd4_sentinel["Date at which cd4 result was issued"] = pd.to_datetime(df_cd4_sentinel["Date at which cd4 result was issued"],errors='coerce').dt.date
        df_cd4_sentinel["Event_date"] = pd.to_datetime(df_cd4_sentinel["Event_date"],errors='coerce').dt.date
        df_cd4_sentinel = df_cd4_sentinel [[
            'Event_date',
            'person_id',
            'Date at which cd4 sample was taken',
            'Date at which cd4 result was issued',
            "CD4 Sample submitted to lab",
            "Was cd4 result issued",
            'CD4 Count',
        ]]
        df_cd4_sentinel = df_cd4_sentinel.sort_values(by=['CD4 Count'])
        df_cd4_sentinel.drop_duplicates(subset=['Event_date','person_id'], keep='last',inplace = True)


        #  Date of Tb diagnosis
        df_TB_diagnosis_sentinel = duckdb.query(
            """
                select date as 'Event_date',
                person_id, 
                date as 'date of TB diagnosis',diagnosis as 'TB diagnosis',
                from person_diagnosis
                where  person_id in (select person_id from merge12)
            """
        ).df()

        df_TB_diagnosis_sentinel['date of TB diagnosis'] = pd.to_datetime(df_TB_diagnosis_sentinel['date of TB diagnosis'],errors='coerce').dt.date
        df_TB_diagnosis_sentinel["Event_date"] = pd.to_datetime(df_TB_diagnosis_sentinel["Event_date"],errors='coerce').dt.date


        #  Date of tb treatment start date
        df_TB_start_date_sentinel = duckdb.query(
            """
                select t.treatment_start_date as 'Event_date',
                tb.person_id, 
                t.treatment_start_date as 'TB Treatment Start Date',
                t.regimen as 'TB Regimen'
                from tb_treatment_current_status t
                left join tb 
                on t.tb_id = tb.tb_id
                where person_id in (select person_id from merge12)
            """
        ).df()

        df_TB_start_date_sentinel['TB Treatment Start Date'] = pd.to_datetime(df_TB_start_date_sentinel['TB Treatment Start Date'],errors='coerce').dt.date
        df_TB_start_date_sentinel["Event_date"] = pd.to_datetime(df_TB_start_date_sentinel["Event_date"],errors='coerce').dt.date



         # Date of tb outcome
        df_TB_outcome_sentinel = duckdb.query(
            """
                select out_come_date as 'Event_date',
                person_id, 
                type_of_tb as 'Type Of TB',
                tb_disease_site as 'TB Disease Site',
                tb_disease_type as 'TB Disease Type', outcome as 'TB Outcome',
                out_come_date as 'Date of TB Outcome'
                from tb
                where  person_id in (select person_id from merge12)
            """
        ).df()

        df_TB_outcome_sentinel['Date of TB Outcome'] = pd.to_datetime(df_TB_outcome_sentinel['Date of TB Outcome'],errors='coerce').dt.date
        df_TB_outcome_sentinel["Event_date"] = pd.to_datetime(df_TB_outcome_sentinel["Event_date"],errors='coerce').dt.date

        df_TB_sentinel2 = pd.merge(df_TB_diagnosis_sentinel,df_TB_start_date_sentinel,on=["person_id","Event_date"],how = "outer")
        df_TB_sentinel = pd.merge(df_TB_sentinel2,df_TB_outcome_sentinel,on=["person_id","Event_date"],how = "outer")
        df_TB_sentinel = df_TB_sentinel[[
            'Event_date',
            'person_id', 'date of TB diagnosis','TB diagnosis',
            'TB Treatment Start Date', 'TB Regimen',
            'TB Outcome','Date of TB Outcome','Type Of TB','TB Disease Site','TB Disease Type'
        ]]
        df_TB_sentinel = df_TB_sentinel.sort_values(by=['TB Outcome'])
        df_TB_sentinel.drop_duplicates(subset=['Event_date','person_id'], keep='last',inplace =  True)
        df_TB_sentinel["Event_date"] = pd.to_datetime(df_TB_sentinel["Event_date"],errors='coerce').dt.date


        df_tpt_eligibility_sentinel = duckdb.query(
            """
                    select 
                    patient.person_id,
                    patient.time as 'Event_date',
                    tpt_screening.eligible as 'Eligible for TPT',
                    sign_and_symptoms_of_active_tb,patient_currently_on_tb_treatment,
                    completed_ipt_in_the_last_three_years,signs_of_active_liver_disease,heavy_alcohol_use,severe_peripheral_neuropathy
                    from tpt_screening 
                    left join patient
                    on tpt_screening.patient_id = patient.patient_id
                    where patient.person_id in (select person_id from merge12)
            """
        ).df()
        df_tpt_eligibility_sentinel = df_tpt_eligibility_sentinel.replace('\x01', 'Yes')
        df_tpt_eligibility_sentinel = df_tpt_eligibility_sentinel.replace('\\0', 'No')
        df_tpt_eligibility_sentinel["Event_date"] = pd.to_datetime(df_tpt_eligibility_sentinel["Event_date"],errors='coerce').dt.date


        df_tpt_eligibility_sentinel["Reason for not being eligible for TPT"] = df_tpt_eligibility_sentinel.apply(extraction.getReasonforNotEiligible, axis =1)

        df_tpt_eligibility_sentinel = df_tpt_eligibility_sentinel[[
            'person_id', 'Event_date','Eligible for TPT', 'Reason for not being eligible for TPT'
        ]]


        df_tpt_not_initiate_sentinel = duckdb.query(
            """
                    select 
                    art.person_id,
                    art_ipt.date as 'Event_date',
                    art_ipt.reason as 'Reason for not initiating TPT'
                    from art_ipt 
                    left join art
                    on art.art_id = art_ipt.art_id
                    where art_ipt.status = 'Not Eligible for IPT' and 
                    art.person_id in (select person_id from merge12)
               
            """
        ).df()
        df_tpt_not_initiate_sentinel["Event_date"] = pd.to_datetime(df_tpt_not_initiate_sentinel["Event_date"],errors='coerce').dt.date
        


        df_tpt_start_sentinel = duckdb.query(
            """
      
                    select 
                    art.person_id,
                    art_ipt.date as 'Event_date',
                    art_ipt.date as "TPT start date", 
                    art_ipt.status as 'TPT Start Regimen'
                    from art_ipt 
                    left join art
                    on art.art_id = art_ipt.art_id
                    left join demographics on
                    art.person_id = demographics.person_id
                    where art_ipt.status in ('3HP Initiated', '3RH Initiated','6H Initiated') and 
                    art.person_id in (select person_id from merge12)
            
            """
        ).df()
        df_tpt_start_sentinel['TPT Start Regimen'] = df_tpt_start_sentinel['TPT Start Regimen'].str.replace('Initiated','')
        df_tpt_start_sentinel["Event_date"] = pd.to_datetime(df_tpt_start_sentinel["Event_date"],errors='coerce').dt.date
   


        df_tpt_stop_sentinel = duckdb.query(
            """
                    select 
                    art.person_id,
                    art.art_id, art_ipt.date as 'Event_date',
                    art_ipt.date as 'Date of stopping TPT',
                    art_ipt.reason as 'Reason for stopping TPT'
                    from art_ipt 
                    left join art
                    on art.art_id = art_ipt.art_id
                    left join demographics on
                    art.person_id = demographics.person_id
                    where art_ipt.status in ('3HP Stopped', 'IPT Stopped','6H Stopped') and 
                    art.person_id in (select person_id from merge12)
              
            """
        ).df()
        df_tpt_stop_sentinel["Event_date"] = pd.to_datetime(df_tpt_stop_sentinel["Event_date"],errors='coerce').dt.date
        


        df_tpt_complete_sentinel = duckdb.query(
            """

                    select 
                    art.person_id,
                    art.art_id,
                    art_ipt.date as 'Event_date',
                    art_ipt.date as "TPT completion date", 
                    art_ipt.status as 'TPT Complete Regimen'
                    from art_ipt 
                    left join art
                    on art.art_id = art_ipt.art_id
                    left join demographics on
                    art.person_id = demographics.person_id
                    where art_ipt.status in ('IPT Completed','3HP Completed', '6H Completed') and 
                    art.person_id in (select person_id from merge12)

            """
        ).df()
        df_tpt_complete_sentinel['TPT Complete Regimen'] = df_tpt_complete_sentinel['TPT Complete Regimen'].str.replace('Completed','')
        df_tpt_complete_sentinel["Event_date"] = pd.to_datetime(df_tpt_complete_sentinel["Event_date"],errors='coerce').dt.date


        sentinel_1 = pd.merge(df_art_visit_sentinel,df_art_current_status_sentinel,on = ["person_id","Event_date"], how ="outer")

        sentinel_2 = pd.merge(sentinel_1,df_art_who_stage_sentinel,on = ["person_id","Event_date"], how ="outer")
        
        sentinel_3 = pd.merge(sentinel_2,df_viral_load_sentinel,on = ["person_id","Event_date"], how ="outer")

        sentinel_4 = pd.merge(sentinel_3,df_cd4_sentinel,on = ["person_id","Event_date"], how ="outer")

        sentinel_5 = pd.merge(sentinel_4,df_TB_sentinel,on = ["person_id","Event_date"], how ="outer")

        sentinel_6 = pd.merge(sentinel_5,df_tpt_eligibility_sentinel,on = ["person_id","Event_date"], how ="outer")

        sentinel_7 = pd.merge(sentinel_6,df_tpt_not_initiate_sentinel,on = ["person_id","Event_date"], how ="outer")
        
        sentinel_9 = pd.merge(sentinel_7,df_tpt_start_sentinel ,on = ["person_id","Event_date"], how ="outer")

        sentinel_10 = pd.merge(sentinel_9,df_tpt_stop_sentinel ,on = ["person_id","Event_date"], how ="outer")

        sentinel_11 = pd.merge(sentinel_10,df_tpt_complete_sentinel ,on = ["person_id","Event_date"], how ="outer")




        sentinel_11 = sentinel_11[[
        'person_id','Event_date', 'Weight at ART Visit',
        'Pregnancy/lactation status', 
       'ARV Status', 'ART Regimen', 'Art Initiation Category', 'Who Stage',
       'Art Follow Up Status', 'Date at which Viral Load result was issued',
       'Date for which Viral Load was taken',
       'Viral Load Sample submitted to lab', 'Was Viral Load result issued',
       'Viral Load result', 'Date at which cd4 sample was taken',
       'Date at which cd4 result was issued', 'CD4 Sample submitted to lab',
       'Was cd4 result issued', 'CD4 Count', 
       'Eligible for TPT',
       'Reason for not being eligible for TPT',
       'Reason for not initiating TPT', 
       'TPT start date', 'TPT Start Regimen', 
       'Date of stopping TPT', 'Reason for stopping TPT', 
       'TPT completion date', 'TPT Complete Regimen', 'date of TB diagnosis', 'TB diagnosis',
       'TB Treatment Start Date','TB Status','TB Regimen',
       'TB Outcome', 'Date of TB Outcome', 'Type Of TB', 'TB Disease Site', 'TB Disease Type'
        ]]

    
        sentinel_11 = sentinel_11.replace(['NULL'], '')
        sentinel_11 = sentinel_11.replace('\x01', 'YES')
        sentinel_11 = sentinel_11.replace('\\0', 'NO')

        static_file = static_file.replace(['NULL'], '')
        static_file = static_file.replace('\x01', 'YES')
        static_file = static_file.replace('\\0', 'NO')

        sentinel_11.to_csv("TPT Sentinel_ " + latest_site_id + ".csv",index = False)

        static_file.to_csv("TPT_static_" + latest_site_id + ".csv",index = False)

        os.remove(filename)
    

    sentinel_files = glob.glob('*.{}'.format('csv'))
    sentinel_files = [file for file in sentinel_files if file.startswith("TPT Sentinel_")]
    df_sentinel =pd.DataFrame()
    for file in sentinel_files:
        df_temp = pd.read_csv(file)
        df_sentinel = df_sentinel.append(df_temp,ignore_index = True)

    
    static_files = glob.glob('*.{}'.format('csv'))
    static_files = [file for file in static_files if file.startswith("TPT_static")]
    df_static =pd.DataFrame()
    for file in static_files:
        df_temp = pd.read_csv(file)
        df_static = df_static.append(df_temp,ignore_index = True)

    
    sequential_number = 0
    def create_TPT_Id(x):
            # tpt id format - site - code (without zw) -gender prefix (m|f)- sequantial number(starts from 1)
            # tpt id Algorithm
            # order list of person_id's from all the processed facilities by person_id
            # for each person_id
                # check if person_id has been assigned a tpt id before (check from a saved dictionary)
                    # if person_id has been assigned tpt id , retrieve the assigned tpt id and assign it to the person_id
                    # else 
                        # extract last last assigned sequential number
                        # generate tpt id where sequential number is last assigned sequantial number + 1
        global sequential_number
        sequential_number += 1
        
        seq_number = str(sequential_number).zfill(6)
        site_code = x["Facility Id"][2:]
        if  pd.isnull(x["sex"]):
             gender = "U"
        else:
             gender = x["sex"][0]
               
        tpt_id = site_code + "-" + gender + "-" + str(seq_number)

        return tpt_id
    
    df_static.sort_values(by = ["person_id"],inplace= True)

    df_static["TPT Id"] = df_static.apply(create_TPT_Id,axis = 1)

    df_staticMaster = df_static [[
            'Province','District','Facility','Facility Id',
            'TPT Id', 'person_id','firstname','lastname','birthdate','sex', 'Client Profile','art_number','Date of HIV diagnosis',
            'Date of enrolmment in care','ART start date','ART initiation category on enrolemnt','ARV Regimen on enrolemnt',
            'Weight at enrolment', 'TB status at enrolment','WHO clinical stage at enrolment',
            'Pregnancy/lactation status at enrolment', 'Eligible for TPT at enrolment','Reason for not being eligible for TPT',
            'TPT start date','TPT Start Regimen','TPT completion date','TPT Complete Regimen',
            'Reason for stopping TPT', 'Date of stopping TPT','date of TB diagnosis', 'TB diagnosis',
            'TB Treatment Start Date','Type Of TB','TB treatment outcome','Date of TB outcome',
             'Follow Up Outcome'
        ]]


    map_dict = dict(zip(df_staticMaster["person_id"], df_staticMaster["TPT Id"]))

    # df_static.drop('person_id', axis=1, inplace=True)

    df_staticFile = df_staticMaster [[
            'Province','District','Facility','Facility Id',
            'TPT Id','birthdate','sex', 'Client Profile','art_number','Date of HIV diagnosis',
            'Date of enrolmment in care','ART start date','ART initiation category on enrolemnt','ARV Regimen on enrolemnt',
            'Weight at enrolment', 'TB status at enrolment','WHO clinical stage at enrolment',
            'Pregnancy/lactation status at enrolment', 'Eligible for TPT at enrolment','Reason for not being eligible for TPT',
            'TPT start date','TPT Start Regimen','TPT completion date','TPT Complete Regimen',
            'Reason for stopping TPT', 'Date of stopping TPT','date of TB diagnosis', 'TB diagnosis',
            'TB Treatment Start Date','Type Of TB','TB treatment outcome','Date of TB outcome',
             'Follow Up Outcome'
        ]]

    df_sentinel["TPT Id"] = df_sentinel["person_id"].map(map_dict)

    df_one = df_sentinel


    df_sentinel = df_sentinel[[
    'person_id','TPT Id','Event_date', 'Weight at ART Visit',
    'Pregnancy/lactation status',  
    'ARV Status', 'ART Regimen', 'Art Initiation Category', 'Who Stage',
    'Art Follow Up Status', 'Date at which Viral Load result was issued',
    'Date for which Viral Load was taken',
    'Viral Load Sample submitted to lab', 'Was Viral Load result issued',
    'Viral Load result', 'Date at which cd4 sample was taken',
    'Date at which cd4 result was issued', 'CD4 Sample submitted to lab',
    'Was cd4 result issued', 'CD4 Count', 
    'Eligible for TPT',
    'Reason for not being eligible for TPT',
    'Reason for not initiating TPT', 
    'TPT start date', 'TPT Start Regimen', 
    'Date of stopping TPT', 'Reason for stopping TPT', 
    'TPT completion date', 'TPT Complete Regimen','date of TB diagnosis', 'TB diagnosis' ,
    'TB Treatment Start Date','TB Status', 'TB Regimen',
    'Type Of TB', 'TB Disease Site', 'TB Disease Type','TB Outcome', 'Date of TB Outcome'
    ]]

    df_sentinel.to_csv("TPT Sentinel_ " + datetime.today().strftime('%d%m%Y') + ".csv",index = False)

    df_staticMaster.to_csv("TPT Static_master_" + datetime.today().strftime('%d%m%Y') + ".csv",index = False)

    df_staticFile.to_csv("TPT Static_" + datetime.today().strftime('%d%m%Y') + ".csv",index = False)



........................................................................
>>> working on  TDH09082023.sql
>>>  Tsholotsho District Hospital
>>> Facility id  ZW05060A
>>> working on sentinel file

........................................................................
>>> working on  Tsungubvi14Sept23.sql
>>>  TSUNGUBVI
>>> Facility id  ZW020447
>>> working on sentinel file

........................................................................
>>> working on  Zengeza300823.sql
>>>  Zengeza Clinic
>>> Facility id  ZW000B04
>>> working on sentinel file
